# Testing Progress — db-rag-assistant

This notebook is used to **demonstrate the progress of the** `db-rag-assistant` project
(RAG for database schema Q&A; stack: Postgres + pgvector + Ollama; UI: Streamlit).

It tests four core components individually (at the unit level) before integrating them into an end-to-end workflow:

1. Connection to Postgres + `pgvector` extension
2. Embedding generation via Ollama
3. Retrieval — similarity search against database schema metadata
4. RAG QA — combining retrieval and generation into a single answer
5. (Bonus) Natural Language → SQL — a next-phase feature currently under development

> Adjust the **CONFIG** section below to match your environment (host, port, DB name, Ollama model).


## 1. Setup & Configuration

In [1]:
# Install dependencies (run only once; skip if already installed)
#!pip install psycopg2-binary pgvector sqlalchemy requests pandas python-dotenv config dotenv openai


In [2]:
import sys
print(sys.executable)

C:\Users\Garjita\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


In [3]:
!python --version

Python 3.13.14


In [4]:
import json
import requests
import pandas as pd
from sqlalchemy import create_engine, text

# ====== CONFIG ======
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "postgres",
}

OLLAMA_BASE_URL = "http://ai_ollama:11434"
EMBED_MODEL = "all-MiniLM-L6-v2"   
LLM_MODEL = "llama3:8b"             

DB_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)
engine = create_engine(DB_URL)
print("Database Config ready!")


Database Config ready!


## 2. Test Database Connection + pgvector

Ensure Postgres is accessible and the `pgvector` extension is enabled—this is a prerequisite for retrieval to work.

In [5]:
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar()
    print("Postgres version:", version)

    ext = conn.execute(text(
        "SELECT extname, extversion FROM pg_extension WHERE extname = 'vector';"
    )).fetchone()
    if ext:
        print(f"pgvector active — version: {ext[1]}")
    else:
        print("WARNING: The pgvector extension is not enabled in this database.")


Postgres version: PostgreSQL 16.4, compiled by Visual C++ build 1940, 64-bit


## 3. Test Generate Embedding (Ollama)

Verify that the local embedding model can be invoked and generates vectors with consistent dimensions.

In [6]:
print(OLLAMA_BASE_URL)

http://ai_ollama:11434


In [7]:
import requests

OLLAMA_BASE_URL = "http://localhost:11434"

r = requests.get(
    f"{OLLAMA_BASE_URL}/api/version",
    timeout=10
)

print("HTTP:", r.status_code)
print(r.text)

HTTP: 200
{"version":"0.32.9"}


In [8]:
r = requests.get(
    f"{OLLAMA_BASE_URL}/api/tags",
    timeout=10
)

print("HTTP:", r.status_code)
print(r.text)

HTTP: 200
{"models":[{"name":"embeddinggemma:latest","model":"embeddinggemma:latest","modified_at":"2026-08-14T01:37:14.466886433Z","size":621875917,"digest":"85462619ee721b466c5927d109d4cb765861907d5417b9109caebc4e614679f1","details":{"parent_model":"embeddinggemma:300m-bf16","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"307.58M","quantization_level":"BF16","context_length":2048,"embedding_length":768},"capabilities":["embedding"]},{"name":"llama3:8b","model":"llama3:8b","modified_at":"2026-08-13T12:32:20.921458994Z","size":4661224676,"digest":"365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_0","context_length":8192,"embedding_length":4096},"capabilities":["completion"]},{"name":"ministral-3:3b","model":"ministral-3:3b","modified_at":"2026-08-13T11:22:06.323989868Z","size":2953840808,"digest":"f04aa1c738f64e13

In [9]:
import requests

OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "embeddinggemma"


def get_embedding(text_input: str, model: str = EMBED_MODEL):
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/embed",
        json={
            "model": model,
            "input": text_input,
        },
        timeout=60,
    )

    resp.raise_for_status()

    return resp.json()["embeddings"][0]


sample_text = "The users table stores customer account data.: id, email, created_at."

vec = get_embedding(sample_text)

print("Embedding vector length:", len(vec))
print("The first 5 values:", vec[:5])

Embedding vector length: 768
The first 5 values: [-0.0752958, 0.028504703, -0.018897155, 0.023013715, -0.009203152]


In [10]:
"""Generation: build a prompt from retrieval results and call the LLM."""
import os
import sys
from openai import OpenAI

from pathlib import Path
PROJECT_ROOT = Path(r"D:\projects\db-rag-assistant-master")
sys.path.insert(0, str(PROJECT_ROOT))
from rag.config import OPENAI_API_KEY, OPENAI_BASE_URL, LLM_MODEL

_client = None

SYSTEM_PROMPT = """You are a database documentation assistant. Answer the
question ONLY based on the provided context. If the answer is not present
in the context, honestly say the information could not be found in the
documentation. Cite the source table/file where relevant. Keep answers
short and technical."""


def _get_client() -> OpenAI:
    global _client
    if _client is None:
        # base_url=None -> defaults to api.openai.com. Set OPENAI_BASE_URL
        # in .env to point at Ollama or any other OpenAI-compatible server.
        _client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
    return _client


def build_prompt(question: str, contexts: list[dict]) -> str:
    context_block = "\n\n".join(
        f"[Source: {c['source_file']}"
        f"{', table: ' + c['table_name'] if c.get('table_name') else ''}]\n"
        f"{c['content']}"
        for c in contexts
    )
    return f"CONTEXT:\n{context_block}\n\nQUESTION: {question}\n\nANSWER:"


def generate_answer(question: str, contexts: list[dict]) -> str:
    prompt = build_prompt(question, contexts)
    client = _get_client()
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content


In [11]:
import inspect

print(inspect.getsource(generate_answer))

def generate_answer(question: str, contexts: list[dict]) -> str:
    prompt = build_prompt(question, contexts)
    client = _get_client()
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content



In [12]:
sample_text = "The users table stores customer account data.: id, email, created_at."

vec = get_embedding(sample_text)

print("Embedding vector length:", len(vec))
print("The first 5 values:", vec[:5])

Embedding vector length: 768
The first 5 values: [-0.0752958, 0.028504703, -0.018897155, 0.023013715, -0.009203152]


## 4. Test Retrieval — Similarity Search

Retrieve several rows of schema metadata (table name + description) from the database, then find the ones most relevant to a user query using pgvector's cosine distance (`<=>`).

> Replace the `schema_metadata` table name with your actual embedding storage schema.

In [13]:
user_question = "Which table stores payment transaction data?n?"
query_vec = get_embedding(user_question)

sql = text('''
    SELECT
    t.table_schema,
    t.table_name,
    obj_description(
        (quote_ident(t.table_schema) || '.' || quote_ident(t.table_name))::regclass,
        'pg_class'
    ) AS description
    FROM information_schema.tables t
    WHERE t.table_schema NOT IN ('pg_catalog', 'information_schema')
    AND t.table_type = 'BASE TABLE'
    ORDER BY t.table_schema, t.table_name;
''')

with engine.connect() as conn:
    rows = conn.execute(sql, {"query_vec": str(query_vec)}).fetchall()

df_retrieval = pd.DataFrame(rows, columns=["table_name", "description", "similarity"])
df_retrieval


,table_name,description,similarity
0,pgagent,pga_exception,None
1,pgagent,pga_job,Job main entry
2,pgagent,pga_jobagent,Active job agents
3,pgagent,pga_jobclass,Job classification
4,pgagent,pga_joblog,Job run logs.
5,pgagent,pga_jobstep,Job step to be executed
6,pgagent,pga_jobsteplog,Job step run logs.
7,pgagent,pga_schedule,Job schedule exceptions
8,public,employees,None
9,public,yellow_tripdata,None


## 5. Test RAG End-to-End (Retrieval + Generation)

Konteks hasil retrieval di atas disuntikkan ke prompt, lalu dikirim ke LLM lokal via Ollama untuk menghasilkan jawaban akhir.

In [14]:
def generate_answer(question: str, context_rows, model: str = LLM_MODEL):
    context_text = "\n".join(
        f"- {r.table_name}: {r.description}" for r in context_rows
    )
    prompt = f'''You are an assistant that answers questions about database schemas.
Use the following context to answer; keep your answers brief and in English.

Context:
{context_text}

Question: {question}
Answer:'''

    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["response"]

    answer = generate_answer(user_question, rows)
    print("QUESTION:", user_question)
    print("\nRAG ANSWER:\n", answer)


## 6. (Bonus) Test Natural Language → SQL

Fitur tahap berikutnya: mengubah pertanyaan bahasa natural menjadi query SQL, memakai konteks skema yang sama
dari hasil retrieval di atas. Bagian ini masih tahap pengembangan — dipakai untuk validasi progress awal.

In [15]:
def generate_sql(question: str, context_rows, model: str = LLM_MODEL):
    context_text = "\n".join(
        f"- {r.table_name}: {r.description}" for r in context_rows
    )
    prompt = f'''You are an assistant that converts natural language questions into PostgreSQL SQL queries.
Output only the SQL query, without any additional explanation.

Relevant scheme:
{context_text}

Question: {question}
SQL:'''

    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["response"].strip()

    generated_sql = generate_sql(user_question, rows)
    print("SQL yang dihasilkan:\n", generated_sql)

# Test execution (comment out or remove if you are certain the query is safe to run)
# with engine.connect() as conn:
#     result = conn.execute(text(generated_sql))
#     display(pd.DataFrame(result.fetchall(), columns=result.keys()))


## 7. Test Result Summary

Fill in this table manually each time you run the notebook to document your progress in the README or GitHub portfolio.

In [16]:
summary = pd.DataFrame([
    {"component": "DB Connection + pgvector", "status": "PASS/FAIL", "notes": ""},
    {"component": "Generate embedding",       "status": "PASS/FAIL", "notes": ""},
    {"component": "Retrieval similarity",     "status": "PASS/FAIL", "notes": ""},
    {"component": "RAG end-to-end",           "status": "PASS/FAIL", "notes": ""},
    {"component": "NL to SQL (bonus)",        "status": "PASS/FAIL", "notes": ""},
    ])
summary

,component,status,notes
0,DB Connection + pgvector,PASS/FAIL,
1,Generate embedding,PASS/FAIL,
2,Retrieval similarity,PASS/FAIL,
3,RAG end-to-end,PASS/FAIL,
4,NL to SQL (bonus),PASS/FAIL,


In [17]:
import requests

# Send requests to Ollama internal endpoints
url = "http://127.0.0.1:11434/api/ps"

try:
    response = requests.get(url).json()
    models = response.get("models", [])
    
    if not models:
        print("There is no model currently active or loaded.")
    else:
        for model in models:
            print(f"Active Model: {model['name']}")
except Exception as e:
    print(f"Connection to Ollama fail: {e}")


Active Model: embeddinggemma:latest
